1. get shape for the correct region

In [1]:
import numpy as np
import holoviews as hv
import hvplot.xarray  # noqa
import hvplot.pandas  # noqa
from rasterio.enums import Resampling

from conflict_monitoring_ntl.case_studies import get_county_ids, get_date
from conflict_monitoring_ntl.satellites import (
    BlackMarbleEE, 
    GHSLSurface, 
    SDGSat, 
    GHSLPopulation,
)
from conflict_monitoring_ntl.viz import plot_tile_comparison, plot_binary
from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import (
    get_gdf_for_admin, 
    get_combined_mask, 
    get_non_nan_flat_array,
    binarize_xarray,
    get_precision_recall
)

In [2]:
country = "Sudan"
date, county_id = get_date(country), get_county_ids(country)[0]
county_gdf = get_gdf_for_admin(county_id)

In [3]:
rasters = [BlackMarbleEE(), SDGSat(), GHSLSurface()]

transformations = [
    {"reproject_match": {"resampling": Resampling.bilinear}}, 
    {"reproject_match": {"resampling": Resampling.bilinear}},
    {}
]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

2025-11-02 11:46:18,145 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-02 11:46:18,159 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-02 11:46:18,163 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-02 11:46:18,171 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-02 11:46:18,223 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-02 11:46:18,245 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.goo

In [ ]:
binary_surface = binarize_xarray(ds.ghsl_surface, 0)
thresholded_binary = binary_surface.where(binary_surface > 0)

ghsl = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#FF0000"],
    title="GHSL Surface (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.5,
    tiles="EsriImagery",
)

In [ ]:
binary_surface = binarize_xarray(ds.black_marble_radiance, 1)
thresholded_binary = binary_surface.where(binary_surface > 0)

black_marble = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#EAFF00"],
    title="Black Marble (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.5,
)

In [ ]:
binary_surface = binarize_xarray(ds.sdgsat_dn, 1.05)
thresholded_binary = binary_surface.where(binary_surface > 0)

sdg_sat = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#00FFEA"],
    title="SDGSat (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.5,
)

In [7]:
ghsl * black_marble * sdg_sat

:Overlay
   .WMTS.I             :WMTS   [Longitude,Latitude]
   .Image.GHSL_Surface :Image   [lon,lat]   (ghsl_surface)
   .Image.Black_Marble :Image   [lon,lat]   (black_marble_radiance)
   .Image.SDGSat       :Image   [lon,lat]   (sdgsat_dn)

## Masisi - DRC

In [8]:
country = "Democratic Republic of the Congo"
date, county_id = get_date(country), get_county_ids(country)[0]
county_gdf = get_gdf_for_admin(county_id)

In [9]:
rasters = [BlackMarbleEE(), SDGSat(), GHSLSurface()]

transformations = [
    {"reproject_match": {"resampling": Resampling.bilinear}}, 
    {"reproject_match": {"resampling": Resampling.bilinear}},
    {}
]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

/Users/jan.kokla/Documents/EPFL/conflict-monitoring-ntl/src/conflict_monitoring_ntl/transform.py:97: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge(processed).rio.write_crs("EPSG:4326")


In [10]:
binary_surface = binarize_xarray(ds.ghsl_surface, 0)
thresholded_binary = binary_surface.where(binary_surface > 0)

ghsl = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#000000", "#FF0000"],
    title="GHSL Surface (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
    tiles='EsriImagery'
)

In [11]:
binary_surface = binarize_xarray(ds.black_marble_radiance, 1)
thresholded_binary = binary_surface.where(binary_surface > 0)

black_marble = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#000000", "#EAFF00"],
    title="Black Marble (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
)

In [12]:
binary_surface = binarize_xarray(ds.sdgsat_dn, 1.5)
thresholded_binary = binary_surface.where(binary_surface > 0)

sdgsat = thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap=["#000000", "#00FFEA"],
    title="SDGSat (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
)

## Rutshuru (ville) - DRC

In [13]:
country = "Democratic Republic of the Congo"
date, county_id = get_date(country), get_county_ids(country)[4]
county_gdf = get_gdf_for_admin(county_id)

In [14]:
rasters = [BlackMarbleEE(), SDGSat(), GHSLSurface()]

transformations = [
    {"reproject_match": {"resampling": Resampling.bilinear}}, 
    {"reproject_match": {"resampling": Resampling.bilinear}},
    {}
]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

/Users/jan.kokla/Documents/EPFL/conflict-monitoring-ntl/src/conflict_monitoring_ntl/transform.py:97: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge(processed).rio.write_crs("EPSG:4326")


In [15]:
binary_surface = binarize_xarray(ds.ghsl_surface, 0)
thresholded_binary = binary_surface.where(binary_surface > 0)

thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap="rainbow",
    title="GHSL Surface (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
    tiles='EsriImagery'
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [lon,lat]   (ghsl_surface)

In [16]:
binary_surface = binarize_xarray(ds.black_marble_radiance, 1)
thresholded_binary = binary_surface.where(binary_surface > 0)

thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap="rainbow",
    title="Black Marble (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
    tiles='EsriImagery'
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [lon,lat]   (black_marble_radiance)

In [17]:
binary_surface = binarize_xarray(ds.sdgsat_dn, 1.05)
thresholded_binary = binary_surface.where(binary_surface > 0)

thresholded_binary.hvplot.image(
    x="lon",
    y="lat",
    crs=binary_surface.rio.crs,
    clim=(0, 1),
    cmap="rainbow",
    title="SDGSat (binary)",
    geo=True,
    colorbar=False,
    height=500,
    width=600,
    alpha=.3,
    tiles='EsriImagery'
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [lon,lat]   (sdgsat_dn)